In [27]:
import numpy as np
import ee
import geemap

ee.Authenticate()
ee.Initialize()

In [10]:
map = geemap.Map()
map.add_basemap("HYBRID")
map.set_center(-122.4194, 37.7749, 10)
map

Map(center=[37.7749, -122.4194], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [ ]:
roi = map.draw_last_feature.geometry()
print("ROI geometry:", roi)

In [22]:
# Creating a timelist
time_start = ee.Date('2005')
time_end = ee.Date('2025')
time_dif = time_end.difference(time_start, 'year').round()
time_list = ee.List.sequence(0, ee.Number(time_dif).subtract(1)).map(
    lambda x: time_start.advance(x, 'year')
)

# Function for obtaining the date for the spicfic time
def annual(date, col):
  start_date = ee.Date(date)
  end_date = start_date.advance(1, 'year')
  img_sum = col.filterDate(start_date, end_date).sum()
  return img_sum.set('system:time_start', start_date.millis())

In [25]:
vis_params4 = {'bands': ['NDVI'], 'palette': ['#f7fcf5', '#c7e9c0', '#74c476', '#238b45', '#00441b'],
              'min': -7970.3, 'max': 87319.55}

In [23]:
# Script in the cell is Demo
ndvi = (
    ee.ImageCollection("MODIS/061/MOD13A2")
    .filterDate(time_start, time_end)
    .select("NDVI")
    .map(lambda x: x.multiply(0.0001).copyProperties(x, ['system:time_start'])
    )
)

ndvi_annual = ee.ImageCollection(time_list.map(lambda x: annual(date=x, col=ndvi)))

In [26]:
map.addLayer(ndvi.mean().clip(roi), vis_params4, "NDVI")